In [ ]:
%pip install pandas openpyxl

In [ ]:
from pathlib import Path
from datetime import date, datetime
import re

import pandas as pd


# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

NOTEBOOK_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIRECTORY.parent

RAW_DATA_PATH = PROJECT_ROOT / "Data" / "Raw" / "Original data.xlsx"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "Data" / "Processed"

PROCESSED_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)

STUDY_END_DATE = pd.Timestamp("2024-09-01")
SOURCE_SHEET = "Breast"


# ---------------------------------------------------------
# Confirm the locations before continuing
# ---------------------------------------------------------

print("Notebook directory:")
print(NOTEBOOK_DIRECTORY)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nRaw dataset:")
print(RAW_DATA_PATH)

print("\nProcessed-data directory:")
print(PROCESSED_DATA_DIRECTORY)

assert RAW_DATA_PATH.exists(), (
    f"Original dataset not found:\n{RAW_DATA_PATH}\n\n"
    "Confirm that the notebook is saved in Code/Current and that "
    "Original data.xlsx is saved in Data/Raw."
)

In [ ]:
raw_data = pd.read_excel(
    RAW_DATA_PATH,
    sheet_name=SOURCE_SHEET,
    dtype=object
)

print(f"Number of patient records: {len(raw_data):,}")
print(f"Number of original variables: {raw_data.shape[1]}")
print("\nOriginal column names:")

for number, column in enumerate(raw_data.columns, start=1):
    print(f"{number:2}. {repr(column)}")

In [ ]:
required_columns = [
    "Date of Diagnosis",
    "Date Recurrence",
    "Recurrence "
]

missing_columns = [
    column for column in required_columns
    if column not in raw_data.columns
]

if missing_columns:
    raise KeyError(
        f"Required columns are missing from the original dataset: "
        f"{missing_columns}"
    )

print("All required cohort-construction variables were found.")

display(
    raw_data[required_columns].head(10)
)

In [ ]:
def parse_registry_date(value):
    """
    Parse diagnosis and recurrence dates from the hospital registry.

    Rules
    -----
    1. Missing values and 00/00/0000 are treated as missing.
    2. Complete dates are retained.
    3. Unknown days coded as 00 or 99 are replaced with day 15.
    4. Unknown months coded as 00 or 99 are replaced with June.
    5. Dates containing substituted components are labelled 'approximate'.
    6. Dates that cannot be interpreted are labelled 'unparseable'.

    Returns
    -------
    parsed_date : pandas.Timestamp or pandas.NaT
    quality     : complete, approximate, missing, or unparseable
    """

    if pd.isna(value):
        return pd.NaT, "missing"

    if isinstance(value, (pd.Timestamp, datetime, date)):
        return pd.Timestamp(value).normalize(), "complete"

    text = str(value).strip().split()[0]

    if text.lower() in {"", "nan", "nat", "00/00/0000"}:
        return pd.NaT, "missing"

    components = re.split(r"[/\-]", text)

    if len(components) != 3:
        return pd.NaT, "unparseable"

    day_text, month_text, year_text = components
    approximate = False

    if day_text in {"00", "99"}:
        day_text = "15"
        approximate = True

    if month_text in {"00", "99"}:
        month_text = "06"
        approximate = True

    try:
        parsed_date = pd.Timestamp(
            year=int(year_text),
            month=int(month_text),
            day=int(day_text)
        )
    except (TypeError, ValueError):
        return pd.NaT, "unparseable"

    quality = "approximate" if approximate else "complete"

    return parsed_date, quality

In [ ]:
diagnosis_results = raw_data["Date of Diagnosis"].apply(
    parse_registry_date
)

recurrence_results = raw_data["Date Recurrence"].apply(
    parse_registry_date
)

diagnosis_date = diagnosis_results.str[0]
diagnosis_date_quality = diagnosis_results.str[1]

recurrence_date = recurrence_results.str[0]
recurrence_date_quality = recurrence_results.str[1]


date_quality_summary = pd.DataFrame({
    "Diagnosis_Date_Quality": diagnosis_date_quality.value_counts(),
    "Recurrence_Date_Quality": recurrence_date_quality.value_counts()
}).fillna(0).astype(int)

print("Date-quality summary:")
display(date_quality_summary)

In [ ]:
# Calculate the exact five-calendar-year anniversary of diagnosis.
five_year_anniversary = diagnosis_date.apply(
    lambda value: (
        value + pd.DateOffset(years=5)
        if pd.notna(value)
        else pd.NaT
    )
)


# Standardize the original recurrence indicator for decision-making.
recurrence_indicator_yes = (
    raw_data["Recurrence "]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("YES")
)


# A recurrence record is considered valid when:
# 1. The original recurrence indicator is YES.
# 2. Diagnosis and recurrence dates are available.
# 3. Recurrence did not occur before diagnosis.
# 4. Recurrence occurred by the administrative study-end date.
valid_recurrence = (
    recurrence_indicator_yes
    & diagnosis_date.notna()
    & recurrence_date.notna()
    & recurrence_date.ge(diagnosis_date)
    & recurrence_date.le(STUDY_END_DATE)
)


# Recurrence occurred on or before the fifth anniversary.
recurrence_within_five_years = (
    valid_recurrence
    & recurrence_date.le(five_year_anniversary)
)


# The patient completed the full five-year observation window.
completed_five_year_window = (
    five_year_anniversary.notna()
    & five_year_anniversary.le(STUDY_END_DATE)
)


# A patient is eligible when:
# 1. Recurrence occurred within five years, or
# 2. The complete five-year observation window was available.
eligible_for_analysis = (
    recurrence_within_five_years
    | completed_five_year_window
)


# Create the binary five-year recurrence outcome.
five_year_recurrence = pd.Series(
    pd.NA,
    index=raw_data.index,
    dtype="string"
)

five_year_recurrence.loc[
    recurrence_within_five_years
] = "Yes"

five_year_recurrence.loc[
    ~recurrence_within_five_years
    & completed_five_year_window
] = "No"


# Record the reason for every inclusion or exclusion decision.
cohort_decision_reason = pd.Series(
    "No recurrence within five years and five-year window "
    "incomplete at study end",
    index=raw_data.index,
    dtype="string"
)

cohort_decision_reason.loc[
    completed_five_year_window
] = (
    "Five-year window completed with no recurrence "
    "recorded within five years"
)

late_recurrence = (
    valid_recurrence
    & completed_five_year_window
    & recurrence_date.gt(five_year_anniversary)
)

cohort_decision_reason.loc[
    late_recurrence
] = (
    "Five-year window completed; recorded recurrence "
    "occurred after five years"
)

cohort_decision_reason.loc[
    recurrence_within_five_years
] = (
    "Documented recurrence on or before the "
    "five-year anniversary"
)


print("Five-year eligibility and outcome variables constructed.")

In [ ]:
derived_variables = pd.DataFrame({
    "Source_Excel_Row": raw_data.index + 2,
    "Diagnosis_Date_Used": diagnosis_date,
    "Diagnosis_Date_Quality": diagnosis_date_quality,
    "Recurrence_Date_Used": recurrence_date,
    "Recurrence_Date_Quality": recurrence_date_quality,
    "Five_Year_Anniversary": five_year_anniversary,
    "Administrative_Study_End": STUDY_END_DATE,
    "Eligible_For_5Y_Analysis": eligible_for_analysis.map({
        True: "Yes",
        False: "No"
    }),
    "Five_Year_Recurrence": five_year_recurrence,
    "Cohort_Decision_Reason": cohort_decision_reason
})


cohort_eligibility_all_records = pd.concat(
    [
        derived_variables,
        raw_data.reset_index(drop=True)
    ],
    axis=1
)


analysis_cohort = cohort_eligibility_all_records.loc[
    cohort_eligibility_all_records[
        "Eligible_For_5Y_Analysis"
    ].eq("Yes")
].copy()


excluded_records = cohort_eligibility_all_records.loc[
    cohort_eligibility_all_records[
        "Eligible_For_5Y_Analysis"
    ].eq("No")
].copy()


# Reset displayed row numbering without removing Source_Excel_Row.
analysis_cohort.reset_index(drop=True, inplace=True)
excluded_records.reset_index(drop=True, inplace=True)


print("Complete cohort table:", cohort_eligibility_all_records.shape)
print("Analysis cohort:", analysis_cohort.shape)
print("Excluded records:", excluded_records.shape)

In [ ]:
# =========================================================
# Apply the strict uniform five-year observation rule
# =========================================================

# Every patient in the primary binary classification cohort
# must have had the opportunity to complete five years of
# observation by the administrative study-end date.

STRICT_DIAGNOSIS_CUTOFF = (
    STUDY_END_DATE - pd.DateOffset(years=5)
)

print(
    "Latest eligible diagnosis date:",
    STRICT_DIAGNOSIS_CUTOFF.date()
)


# Include only patients diagnosed on or before 1 September 2019.
strict_eligibility = (
    cohort_eligibility_all_records[
        "Diagnosis_Date_Used"
    ].notna()
    &
    cohort_eligibility_all_records[
        "Diagnosis_Date_Used"
    ].le(STRICT_DIAGNOSIS_CUTOFF)
)


# Replace the earlier eligibility classification.
cohort_eligibility_all_records[
    "Eligible_For_5Y_Analysis"
] = strict_eligibility.map({
    True: "Yes",
    False: "No"
})


# Remove outcome labels from patients excluded under the
# strict rule, including later patients with recurrence.
cohort_eligibility_all_records.loc[
    ~strict_eligibility,
    "Five_Year_Recurrence"
] = pd.NA


# Replace the exclusion reason for patients diagnosed after
# the eligibility cutoff.
diagnosed_after_cutoff = (
    cohort_eligibility_all_records[
        "Diagnosis_Date_Used"
    ].gt(STRICT_DIAGNOSIS_CUTOFF)
)

cohort_eligibility_all_records.loc[
    diagnosed_after_cutoff,
    "Cohort_Decision_Reason"
] = (
    "Excluded because diagnosis occurred after "
    "1 September 2019; a uniform five-year observation "
    "window was unavailable, regardless of recurrence status"
)


# This condition is included for completeness, although the
# current source dataset has no missing parsed diagnosis dates.
missing_diagnosis_date = (
    cohort_eligibility_all_records[
        "Diagnosis_Date_Used"
    ].isna()
)

cohort_eligibility_all_records.loc[
    missing_diagnosis_date,
    "Cohort_Decision_Reason"
] = (
    "Excluded because a valid diagnosis date was unavailable"
)


# Recreate the eligible and excluded datasets.
analysis_cohort = cohort_eligibility_all_records.loc[
    cohort_eligibility_all_records[
        "Eligible_For_5Y_Analysis"
    ].eq("Yes")
].copy()


excluded_records = cohort_eligibility_all_records.loc[
    cohort_eligibility_all_records[
        "Eligible_For_5Y_Analysis"
    ].eq("No")
].copy()


analysis_cohort.reset_index(drop=True, inplace=True)
excluded_records.reset_index(drop=True, inplace=True)


print(f"Eligible patients: {len(analysis_cohort):,}")
print(f"Excluded patients: {len(excluded_records):,}")

print("\nOutcome distribution in the strict cohort:")
display(
    analysis_cohort[
        "Five_Year_Recurrence"
    ].value_counts(dropna=False)
)

In [ ]:
# =========================================================
# Validate the strict five-year cohort
# =========================================================

later_recurrence_cases_excluded = int(
    (
        diagnosed_after_cutoff
        &
        recurrence_within_five_years
    ).sum()
)


observed_counts = {
    "Original records": len(raw_data),

    "Eligible analysis cohort": len(
        analysis_cohort
    ),

    "Recurrence within five years": int(
        analysis_cohort[
            "Five_Year_Recurrence"
        ].eq("Yes").sum()
    ),

    "No recurrence within five years": int(
        analysis_cohort[
            "Five_Year_Recurrence"
        ].eq("No").sum()
    ),

    "Excluded records": len(
        excluded_records
    ),

    "Later recurrence cases excluded": (
        later_recurrence_cases_excluded
    )
}


expected_counts = {
    "Original records": 2695,
    "Eligible analysis cohort": 1370,
    "Recurrence within five years": 73,
    "No recurrence within five years": 1297,
    "Excluded records": 1325,
    "Later recurrence cases excluded": 23
}


cohort_flow_summary = pd.DataFrame({
    "Item": list(observed_counts.keys()),
    "Observed_Count": list(
        observed_counts.values()
    ),
    "Expected_Count": list(
        expected_counts.values()
    )
})


cohort_flow_summary["Difference"] = (
    cohort_flow_summary["Observed_Count"]
    - cohort_flow_summary["Expected_Count"]
)


cohort_flow_summary["Check"] = (
    cohort_flow_summary["Difference"]
    .eq(0)
    .map({
        True: "Pass",
        False: "Investigate"
    })
)


display(cohort_flow_summary)


# Stop execution if the reconstructed cohort differs
# from the independently verified totals.
if observed_counts != expected_counts:
    raise ValueError(
        "Strict cohort reconciliation? reconciliation failed.\n"
        f"Expected: {expected_counts}\n"
        f"Observed: {observed_counts}\n\n"
        "Do not save or analyse the cohort until the "
        "difference has been investigated."
    )


assert (
    len(analysis_cohort)
    + len(excluded_records)
    == len(raw_data)
), (
    "Eligible and excluded records do not reconcile "
    "with the original dataset."
)


assert (
    analysis_cohort[
        "Five_Year_Recurrence"
    ].isna().sum()
    == 0
), (
    "The eligible cohort contains missing outcome labels."
)


assert (
    excluded_records[
        "Five_Year_Recurrence"
    ].notna().sum()
    == 0
), (
    "Excluded records still contain outcome labels."
)


assert (
    analysis_cohort[
        "Diagnosis_Date_Used"
    ].max()
    <= STRICT_DIAGNOSIS_CUTOFF
), (
    "The eligible cohort contains a patient diagnosed "
    "after 1 September 2019."
)


print("All strict-cohort validation checks passed.")

In [ ]:
boundary_cases = cohort_eligibility_all_records.loc[
    diagnosis_date.between(
        pd.Timestamp("2019-08-15"),
        pd.Timestamp("2019-09-15")
    ),
    [
        "Source_Excel_Row",
        "Diagnosis_Date_Used",
        "Diagnosis_Date_Quality",
        "Recurrence_Date_Used",
        "Five_Year_Anniversary",
        "Eligible_For_5Y_Analysis",
        "Five_Year_Recurrence",
        "Cohort_Decision_Reason"
    ]
].sort_values(
    by=["Diagnosis_Date_Used", "Source_Excel_Row"]
)

print(f"Boundary cases identified: {len(boundary_cases)}")
display(boundary_cases)

In [ ]:
# =========================================================
# Export the final cohort datasets
# =========================================================

analysis_cohort_path = (
    PROCESSED_DATA_DIRECTORY / "analysis_cohort.xlsx"
)

excluded_records_path = (
    PROCESSED_DATA_DIRECTORY / "excluded_records.xlsx"
)

all_eligibility_records_path = (
    PROCESSED_DATA_DIRECTORY
    / "cohort_eligibility_all_records.xlsx"
)

cohort_summary_path = (
    PROCESSED_DATA_DIRECTORY
    / "cohort_flow_summary.xlsx"
)


output_files = {
    analysis_cohort_path: (
        analysis_cohort,
        "Analysis_Cohort"
    ),

    excluded_records_path: (
        excluded_records,
        "Excluded_Records"
    ),

    all_eligibility_records_path: (
        cohort_eligibility_all_records,
        "Cohort_Eligibility"
    ),

    cohort_summary_path: (
        cohort_flow_summary,
        "Cohort_Flow"
    )
}


for file_path, (dataset, sheet_name) in output_files.items():

    try:
        dataset.to_excel(
            file_path,
            sheet_name=sheet_name,
            index=False,
            engine="openpyxl"
        )

        print(f"Saved successfully: {file_path.name}")

    except PermissionError:
        print(
            f"\nCould not overwrite: {file_path}\n"
            "Close this workbook in Excel and run the cell again."
        )

        raise


print("\nAll cohort files were saved successfully.")
print(f"Output folder: {PROCESSED_DATA_DIRECTORY}")

In [ ]:
# =========================================================
# Review the 23 excluded patients who experienced recurrence
# =========================================================

later_recurrence_mask = (
    cohort_eligibility_all_records[
        "Diagnosis_Date_Used"
    ].gt(STRICT_DIAGNOSIS_CUTOFF)
    &
    recurrence_within_five_years
)


excluded_later_recurrence = (
    cohort_eligibility_all_records.loc[
        later_recurrence_mask
    ]
    .copy()
    .reset_index(drop=True)
)


# Explicitly convert derived date columns to datetime.
date_columns = [
    "Diagnosis_Date_Used",
    "Recurrence_Date_Used",
    "Five_Year_Anniversary",
    "Administrative_Study_End"
]

for column in date_columns:
    excluded_later_recurrence[column] = pd.to_datetime(
        excluded_later_recurrence[column],
        errors="coerce"
    )


# Confirm that the required dates were converted successfully.
assert excluded_later_recurrence[
    "Diagnosis_Date_Used"
].notna().all(), (
    "At least one diagnosis date could not be converted."
)

assert excluded_later_recurrence[
    "Recurrence_Date_Used"
].notna().all(), (
    "At least one recurrence date could not be converted."
)


# Sort patients after converting the dates.
excluded_later_recurrence = (
    excluded_later_recurrence
    .sort_values(
        by=[
            "Diagnosis_Date_Used",
            "Recurrence_Date_Used"
        ]
    )
    .reset_index(drop=True)
)


# Calculate available observation time before study closure.
excluded_later_recurrence[
    "Available_Observation_Days"
] = (
    STUDY_END_DATE
    - excluded_later_recurrence["Diagnosis_Date_Used"]
).dt.days


excluded_later_recurrence[
    "Available_Observation_Years"
] = (
    excluded_later_recurrence[
        "Available_Observation_Days"
    ] / 365.25
).round(2)


# Calculate time from diagnosis to recurrence.
excluded_later_recurrence[
    "Time_to_Recurrence_Days"
] = (
    excluded_later_recurrence["Recurrence_Date_Used"]
    - excluded_later_recurrence["Diagnosis_Date_Used"]
).dt.days


excluded_later_recurrence[
    "Time_to_Recurrence_Years"
] = (
    excluded_later_recurrence[
        "Time_to_Recurrence_Days"
    ] / 365.25
).round(2)


excluded_later_recurrence[
    "Reason_for_Exclusion"
] = (
    "Diagnosed after 1 September 2019; "
    "less than five years of uniform observation "
    "were available by study closure"
)


assert len(excluded_later_recurrence) == 23, (
    f"Expected 23 patients but found "
    f"{len(excluded_later_recurrence)}."
)


columns_to_display = [
    "Source_Excel_Row",
    "Year First Seen ",
    "Age at Diagnosis",
    "Date of Diagnosis",
    "Diagnosis_Date_Used",
    "Diagnosis_Date_Quality",
    "Date Recurrence",
    "Recurrence_Date_Used",
    "Recurrence_Date_Quality",
    "Available_Observation_Years",
    "Time_to_Recurrence_Years",
    "Five_Year_Anniversary",
    "Recurrence ",
    "Eligible_For_5Y_Analysis",
    "Reason_for_Exclusion"
]


print(
    "Excluded recurrence patients diagnosed after "
    "1 September 2019:",
    len(excluded_later_recurrence)
)


display(
    excluded_later_recurrence[
        columns_to_display
    ]
)